# _lib/derivation_helpers — Gold wide-table derivation helpers

Pure helper functions for the time and line-level derived columns on
`DATAMART.sales_lines_wide`.

**Import via `%run ./_lib/derivation_helpers` from the Gold DLT pipeline notebook.**

| Function | Returns | Used for |
|---|---|---|
| `derive_order_date(col)` | `DateType` | `CAST(order_ts AS DATE)` |
| `derive_order_hour(col)` | `IntegerType` | `HOUR(order_ts)` (0–23) |
| `derive_order_day_of_week(col)` | `StringType` | `DATE_FORMAT(order_ts, 'EEEE')` (e.g. 'Monday') |
| `derive_order_year_month(col)` | `DateType` | `DATE_TRUNC('month', order_ts)` (first-of-month DATE) |
| `derive_shift_duration_minutes(start_col, end_col)` | `IntegerType` | `(end_seconds - start_seconds) / 60` from `'HH:mm:ss'` strings |
| `derive_line_subtotal(qty_col, price_col)` | `DecimalType(38, 4)` | `quantity * unit_price` |

All functions return a Spark `Column` expression — no side effects, no Spark context
dependency in the function signatures themselves (Column objects are lazy).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Column


def derive_order_date(order_ts_col: Column) -> Column:
    """Cast a TimestampType order_ts column to DateType.

    Example: 2024-03-15 14:30:00  ->  2024-03-15
    """
    return order_ts_col.cast("date")


def derive_order_hour(order_ts_col: Column) -> Column:
    """Extract the hour-of-day (0–23) from a TimestampType order_ts column.

    Returned as IntegerType.  Enables filtering by hour bucket in dashboards
    and Genie without a re-derivation on every query.

    Example: 2024-03-15 14:30:00  ->  14
    """
    return F.hour(order_ts_col).cast("int")


def derive_order_day_of_week(order_ts_col: Column) -> Column:
    """Extract the full day-of-week name from a TimestampType order_ts column.

    Uses Spark's EEEE locale-sensitive format pattern.  Returns a StringType
    value like 'Monday', 'Tuesday', etc.  The Databricks cluster locale
    defaults to English.

    Example: 2024-03-15 (Friday)  ->  'Friday'
    """
    return F.date_format(order_ts_col, "EEEE")


def derive_order_year_month(order_ts_col: Column) -> Column:
    """Truncate a TimestampType order_ts column to the first day of its month.

    Returns a DateType value (first-of-month DATE).  Enables month-level
    filtering and grouping in dashboards without string-parsing.

    Example: 2024-03-15 14:30:00  ->  2024-03-01
    """
    return F.date_trunc("month", order_ts_col).cast("date")


def _hhmmss_to_seconds(hhmmss_col: Column) -> Column:
    """Convert a 'HH:mm:ss' StringType column to seconds-since-midnight as IntegerType.

    Silver stores shift_start_time and shift_end_time as 'HH:mm:ss' strings.
    This helper is internal to derivation_helpers; callers should use
    derive_shift_duration_minutes.

    Example: '10:30:45'  ->  37845
    """
    hh = F.split(hhmmss_col, ":").getItem(0).cast("int")
    mm = F.split(hhmmss_col, ":").getItem(1).cast("int")
    ss = F.split(hhmmss_col, ":").getItem(2).cast("int")
    return (hh * 3600 + mm * 60 + ss)


def derive_shift_duration_minutes(shift_start_col: Column, shift_end_col: Column) -> Column:
    """Compute shift duration in whole minutes from two 'HH:mm:ss' StringType columns.

    Silver's shift_start_time and shift_end_time are 'HH:mm:ss' strings.
    This function parses both to seconds-since-midnight and returns the
    difference in minutes, cast to IntegerType (integer division — seconds
    remainder is dropped).

    Returns NULL when either input is NULL (natural Spark NULL propagation).

    Example: shift_start='08:00:00', shift_end='10:30:00'  ->  150
    """
    start_seconds = _hhmmss_to_seconds(shift_start_col)
    end_seconds   = _hhmmss_to_seconds(shift_end_col)
    return ((end_seconds - start_seconds) / 60).cast("int")


def derive_line_subtotal(quantity_col: Column, unit_price_col: Column) -> Column:
    """Compute line subtotal as quantity * unit_price, cast to DECIMAL(38, 4).

    Provides a sanity-check column against the Bronze `price` column —
    analysts can compare `line_subtotal` vs `price` to detect pricing anomalies
    without recomputing the expression on every query.

    Returns NULL when either input is NULL (natural Spark NULL propagation).

    Example: quantity=3, unit_price=4.5000  ->  13.5000
    """
    return (quantity_col * unit_price_col).cast("decimal(38, 4)")


print(
    "derivation_helpers loaded: "
    "derive_order_date, derive_order_hour, derive_order_day_of_week, "
    "derive_order_year_month, derive_shift_duration_minutes, derive_line_subtotal"
)